# 07.2 — Fine-tuning BERT

**The fine-tuning paradigm:** Take pre-trained BERT, add a small task-specific head, train on labeled data for a few epochs. The pre-trained representations do most of the work — fine-tuning just specializes them.

**Tasks covered:**
1. Sequence classification (e.g., sentiment) — linear layer on [CLS]
2. Token classification (e.g., NER) — linear layer on all tokens
3. Question answering — predict start/end span

**From scratch first**, then we'll see how HuggingFace wraps this.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Reuse our mini BERT from notebook 07.1
class BERTEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.seg_emb = nn.Embedding(2, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, tokens, segments=None):
        B, T = tokens.shape
        pos = torch.arange(T, device=tokens.device).unsqueeze(0)
        if segments is None:
            segments = torch.zeros_like(tokens)
        return self.drop(self.norm(self.tok_emb(tokens) + self.pos_emb(pos) + self.seg_emb(segments)))

class BERTLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        d_k = d_model // n_heads
        self.n_heads, self.d_k = n_heads, d_k
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        B, T, D = x.shape
        def split(t): return t.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        Q, K, V = split(self.W_Q(x)), split(self.W_K(x)), split(self.W_V(x))
        scores = (Q @ K.transpose(-2,-1)) / math.sqrt(self.d_k)
        if mask is not None: scores = scores.masked_fill(mask==0, float('-inf'))
        attn = self.drop(F.softmax(scores, dim=-1))
        out = (attn @ V).transpose(1,2).contiguous().view(B, T, D)
        x = self.norm1(x + self.drop(self.W_O(out)))
        x = self.norm2(x + self.ff(x))
        return x

class BERTModel(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=4, d_ff=256, max_len=128, dropout=0.1):
        super().__init__()
        self.embedding = BERTEmbedding(vocab_size, d_model, max_len, dropout)
        self.layers = nn.ModuleList([BERTLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.pool = nn.Linear(d_model, d_model)
        self.d_model = d_model
    def forward(self, tokens, segments=None, mask=None):
        x = self.embedding(tokens, segments)
        for layer in self.layers:
            x = layer(x, mask)
        cls_repr = torch.tanh(self.pool(x[:, 0]))
        return x, cls_repr

print("Mini-BERT backbone defined.")

In [ ]:
# ---- Task 1: Sequence Classification ----
# Add a linear layer on top of [CLS] representation.
# This is how sentiment analysis, topic classification, NLI work.

class BERTForSequenceClassification(nn.Module):
    """
    BERT + linear head on [CLS] for text classification.
    During fine-tuning: update ALL weights (BERT + head).
    Typical learning rate: 2e-5 (much smaller than pre-training 1e-4).
    """
    def __init__(self, bert: BERTModel, n_classes: int, dropout=0.1):
        super().__init__()
        self.bert = bert
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(bert.d_model, n_classes)
    
    def forward(self, tokens, segments=None, mask=None, labels=None):
        _, cls_repr = self.bert(tokens, segments, mask)
        logits = self.classifier(self.drop(cls_repr))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
        return logits, loss


# Synthetic sentiment dataset
VOCAB = 500
CLS_ID, SEP_ID = 1, 2

def make_sentiment_batch(batch_size=16, seq_len=20):
    tokens = torch.randint(3, VOCAB, (batch_size, seq_len))
    tokens[:, 0] = CLS_ID
    tokens[:, -1] = SEP_ID
    labels = torch.randint(0, 2, (batch_size,))  # binary: 0=neg, 1=pos
    return tokens, labels

bert = BERTModel(vocab_size=VOCAB, d_model=64, n_heads=4, n_layers=2)
clf = BERTForSequenceClassification(bert, n_classes=2)
optimizer = torch.optim.AdamW(clf.parameters(), lr=2e-4, weight_decay=0.01)

print("Training sentiment classifier...")
for step in range(200):
    tokens, labels = make_sentiment_batch(32, 20)
    logits, loss = clf(tokens, labels=labels)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(clf.parameters(), 1.0)
    optimizer.step()
    if step % 50 == 0:
        acc = (logits.argmax(-1) == labels).float().mean()
        print(f"  step {step:3d}: loss={loss.item():.4f}  acc={acc:.3f}")

print("\nNote: synthetic random data, so ~50% is ceiling. Real data would converge higher.")

In [ ]:
# ---- Task 2: Token Classification (NER) ----
# Instead of pooling to [CLS], apply a linear layer at EVERY token position.
# Labels: B-PER, I-PER, B-ORG, I-ORG, O, etc. (BIO scheme)

class BERTForTokenClassification(nn.Module):
    """
    BERT + per-token linear head for NER, POS, chunking.
    The key difference from sequence classification: we use ALL token reprs,
    not just [CLS].
    """
    def __init__(self, bert: BERTModel, n_labels: int, dropout=0.1):
        super().__init__()
        self.bert = bert
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(bert.d_model, n_labels)
    
    def forward(self, tokens, segments=None, mask=None, labels=None):
        all_repr, _ = self.bert(tokens, segments, mask)
        logits = self.classifier(self.drop(all_repr))  # [B, T, n_labels]
        loss = None
        if labels is not None:
            # ignore_index=-100 for padding and [CLS]/[SEP] tokens
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), 
                                   labels.view(-1), ignore_index=-100)
        return logits, loss


# BIO NER labels: O=0, B-PER=1, I-PER=2, B-ORG=3, I-ORG=4
NER_LABELS = 5

bert2 = BERTModel(vocab_size=VOCAB, d_model=64, n_heads=4, n_layers=2)
ner_model = BERTForTokenClassification(bert2, n_labels=NER_LABELS)

# Synthetic NER batch
seq_len = 20
ner_tokens = torch.randint(3, VOCAB, (4, seq_len))
ner_tokens[:, 0] = CLS_ID
ner_tokens[:, -1] = SEP_ID

ner_labels = torch.randint(0, NER_LABELS, (4, seq_len))
ner_labels[:, 0] = -100   # ignore [CLS]
ner_labels[:, -1] = -100  # ignore [SEP]

logits, loss = ner_model(ner_tokens, labels=ner_labels)
print(f"NER logits: {logits.shape}  (B, T, n_labels)")
print(f"NER loss: {loss.item():.4f}")
print()

# BIO constraint: after B-PER, only I-PER or O is valid
print("BIO constraint: models often violate this (predict I-X after O).")
print("Fix: post-process with Viterbi (CRF head) or just mask invalid transitions.")
print("In practice: BERT + linear head often good enough without CRF.")

In [ ]:
# ---- Fine-tuning best practices ----

print("=== Fine-tuning Best Practices ===")
print()
print("1. Learning Rate")
print("   Pre-training: 1e-4 (train from scratch, large LR OK)")
print("   Fine-tuning:  2e-5 to 5e-5 (small LR to not destroy pre-trained weights)")
print("   Head only:    1e-3 (if freezing BERT and only training the head)")
print()
print("2. Warmup + Linear Decay")
print("   Warmup for 10% of steps, then linearly decay to 0.")
print("   Prevents large gradient updates at the start.")
print()
print("3. Batch size")
print("   Devlin et al.: 16 or 32. GPU memory is the real constraint.")
print("   Gradient accumulation: simulate large batch on small GPU.")
print()
print("4. Epochs")
print("   Usually 2-4 epochs. More leads to catastrophic forgetting.")
print("   Monitor val loss — stop early.")
print()
print("5. Layer-wise Learning Rate Decay (LLRD)")
print("   Later layers = higher LR, earlier layers = lower LR.")
print("   Early layers encode syntax (stable), later = task-specific.")
print("   Multiplier: lr_layer_k = lr_base * decay^(n_layers - k)")
print()

# LLRD implementation
def get_llrd_optimizer(model, base_lr=2e-5, decay=0.9):
    layers = model.bert.layers
    n = len(layers)
    param_groups = []
    for i, layer in enumerate(layers):
        lr = base_lr * (decay ** (n - i))
        param_groups.append({'params': layer.parameters(), 'lr': lr})
    # Embedding + head at base LR
    param_groups.append({'params': model.bert.embedding.parameters(), 'lr': base_lr * (decay**n)})
    param_groups.append({'params': model.classifier.parameters(), 'lr': base_lr})
    return torch.optim.AdamW(param_groups)

llrd_opt = get_llrd_optimizer(clf)
lrs = [g['lr'] for g in llrd_opt.param_groups]
print(f"LLRD learning rates (low to high): {[f'{lr:.2e}' for lr in lrs]}")

In [ ]:
# ---- HuggingFace: the same thing in 10 lines ----
# In production you'd use HuggingFace, not a from-scratch BERT.
# Understand what it's doing under the hood (now you do).

print("""HuggingFace equivalent:

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Tokenize your data
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')

# TrainingArguments controls LR, warmup, epochs, batch_size
args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    per_device_train_batch_size=16,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
trainer.train()
"""
)

print("What HuggingFace gives you on top:")
print("  - Pre-trained weights (bert-base-uncased, roberta-base, etc.)")
print("  - WordPiece tokenizer (handles subwords, OOV)")
print("  - Mixed precision training (fp16)")
print("  - Gradient checkpointing (trade compute for memory)")
print("  - Evaluation + checkpointing during training")

## Summary

**Fine-tuning paradigm:**
- Sequence classification: `[CLS]` repr → linear layer
- Token classification: all token reprs → linear layer per token
- QA: predict start/end span indices over passage tokens

**The magic:** BERT's representations are already rich enough that a linear layer on top usually works. The fine-tuning just steers the representations toward the task.

**What breaks it:** Domain shift. BERT trained on Wikipedia/Books struggles on medical text, code, or tweets. Solution: domain-adaptive pre-training (continue MLM on your domain before fine-tuning).

---
**Next:** 07.3 — Sentence Embeddings + FAISS